# personal-lm 云端训练（从零 GPT，GPU）

**数据流**：Google Drive（ostore gdrive 同步或手动上传）→ 本 notebook 训练 → bundle 写回 Drive → mac 拉回部署。

**隐私**：语料是个人数据，只存你自己的 Drive 与本次 VM 内存；不落任何第三方。

**运行时**：代码执行程序 → 选 GPU（A100/L4/T4 都能跑，A100 最快）。

模型：decoder-only GPT，12 层 / d=768 / ~98M 参数（本地 MPS 训不动的规模，云上半小时级）。

In [ ]:
# 1) GPU 环境确认
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| bf16:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else '-')

In [ ]:
# 2) 挂载 Drive + 数据检查
from google.colab import drive
drive.mount('/content/drive')
import os
DATA_DIR = '/content/drive/MyDrive/clipvault-personal-lm'
os.makedirs(DATA_DIR, exist_ok=True)
!ls -la {DATA_DIR}/ || echo '目录为空：上传 tokens_weighted.bin 与 tokenizer.json（或 ostore gdrive 同步）'

In [ ]:
# 2.5) （gdrive 未同步时的手动 fallback）直接上传两个文件
# tokens_weighted.bin + tokenizer.json（本地: personal-lm/corpus/ 下）
# 已有文件则跳过此格
# from google.colab import files
# files.upload()  # 然后移动到 DATA_DIR

In [ ]:
# 3) 写训练/模型代码（单文件，无仓库依赖）
%%writefile /content/pl_train.py
import argparse, json, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

class CausalSelfAttention(nn.Module):
    def __init__(self, d, nh, block, drop):
        super().__init__()
        assert d % nh == 0
        self.nh, self.dh = nh, d // nh
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.drop = drop
        self.resid_drop = nn.Dropout(drop)
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.nh, self.dh).transpose(1, 2)
        k = k.view(B, T, self.nh, self.dh).transpose(1, 2)
        v = v.view(B, T, self.nh, self.dh).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
            dropout_p=self.drop if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y))

class MLP(nn.Module):
    def __init__(self, d, drop):
        super().__init__()
        self.fc = nn.Linear(d, 4 * d)
        self.proj = nn.Linear(4 * d, d)
        self.drop = nn.Dropout(drop)
    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x))))

class Block(nn.Module):
    def __init__(self, d, nh, block, drop):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = CausalSelfAttention(d, nh, block, drop)
        self.ln2 = nn.LayerNorm(d)
        self.mlp = MLP(d, drop)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab, block, n_layer, n_head, d_model, drop=0.0):
        super().__init__()
        self.block_size = block
        self.config = dict(vocab_size=vocab, block_size=block, n_layer=n_layer,
                           n_head=n_head, d_model=d_model)
        self.tok_emb = nn.Embedding(vocab, d_model)
        self.pos_emb = nn.Embedding(block, d_model)
        self.drop = nn.Dropout(drop)
        self.blocks = nn.ModuleList([Block(d_model, n_head, block, drop) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab, bias=False)
        self.apply(self._init)
    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for blk in self.blocks: x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss
    def num_params(self):
        return sum(p.numel() for p in self.parameters())

def get_batch(data, block, batch, device, rng):
    ix = rng.integers(0, len(data) - block - 1, size=batch)
    x = np.stack([data[i:i+block] for i in ix])
    y = np.stack([data[i+1:i+block+1] for i in ix])
    to = lambda a: torch.from_numpy(a.astype(np.int64)).to(device)
    return to(x), to(y)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--data-dir', default='/content/drive/MyDrive/clipvault-personal-lm')
    ap.add_argument('--steps', type=int, default=6000)
    ap.add_argument('--batch', type=int, default=32)
    ap.add_argument('--block', type=int, default=512)
    ap.add_argument('--lr', type=float, default=6e-4)
    ap.add_argument('--layers', type=int, default=12)
    ap.add_argument('--d-model', type=int, default=768)
    ap.add_argument('--heads', type=int, default=12)
    args = ap.parse_args()
    dd = Path(args.data_dir)
    device = 'cuda'
    data = np.memmap(dd / 'tokens_weighted.bin', dtype=np.uint16, mode='r')
    n_val = max(1000, len(data) // 100)
    val, train = data[-n_val:], data[:-n_val]
    tok = json.loads((dd / 'tokenizer.json').read_text())
    vocab = len(tok['vocab'])
    model = GPT(vocab, args.block, args.layers, args.heads, args.d_model).to(device)
    print(f'params: {model.num_params()/1e6:.1f}M | train tokens: {len(train):,}')
    ckpt_p = dd / 'ckpt_cloud.pt'
    step0 = 0
    opt = None
    if ckpt_p.exists():
        payload = torch.load(ckpt_p, map_location=device, weights_only=False)
        model.load_state_dict(payload['model'])
        step0 = payload['step']
        opt = torch.optim.AdamW(model.parameters(), lr=args.lr, betas=(0.9, 0.95), weight_decay=0.1)
        opt.load_state_dict(payload['opt'])
        print(f'resumed @ step {step0}')
    if opt is None:
        opt = torch.optim.AdamW(model.parameters(), lr=args.lr, betas=(0.9, 0.95), weight_decay=0.1)
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler('cuda', enabled=(amp_dtype == torch.float16))
    rng = np.random.default_rng(1337)
    warmup = 200
    t0 = time.time()
    for step in range(step0, args.steps):
        lr = args.lr * (step + 1) / warmup if step < warmup else \
             args.lr * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(1.0, (step - warmup) / max(1, args.steps - warmup)))))
        for g in opt.param_groups: g['lr'] = lr
        xb, yb = get_batch(train, args.block, args.batch, device, rng)
        with torch.autocast('cuda', dtype=amp_dtype):
            _, loss = model(xb, yb)
        opt.zero_grad(set_to_none=True)
        if scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        if step % 100 == 0 or step == args.steps - 1:
            tps = (step - step0 + 1) * args.batch * args.block / max(time.time() - t0, 1e-6)
            print(f'step {step}/{args.steps} loss {loss.item():.4f} lr {lr:.6f} {tps:,.0f} tok/s')
        if (step + 1) % 500 == 0 or step == args.steps - 1:
            model.eval(); vl = []
            with torch.no_grad():
                for _ in range(8):
                    xb, yb = get_batch(val, args.block, args.batch, device, rng)
                    with torch.autocast('cuda', dtype=amp_dtype):
                        _, l = model(xb, yb)
                    vl.append(l.item())
            model.train()
            v = sum(vl) / len(vl)
            print(f'val loss {v:.4f} (ppl {math.exp(v):.1f})')
            torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                        'step': step + 1, 'config': model.config}, ckpt_p)
    print('done')

if __name__ == '__main__':
    main()


In [ ]:
# 4) 训练（~98M 参数，A100 约 30-60 分钟；中断重跑自动续）
!python /content/pl_train.py --steps 6000 --batch 32

In [ ]:
# 5) 导出 bundle 到 Drive（mac 端 ostore pull 回来即可部署）
import json, torch, shutil
from safetensors.torch import save_file
from pathlib import Path
dd = Path('/content/drive/MyDrive/clipvault-personal-lm')
payload = torch.load(dd / 'ckpt_cloud.pt', map_location='cpu', weights_only=False)
out = dd / 'exports' / f"v{payload['step']}"
out.mkdir(parents=True, exist_ok=True)
sd = {k: v.to(torch.float32).contiguous() for k, v in payload['model'].items()}
save_file(sd, out / 'model.safetensors', metadata={'format': 'pt'})
(out / 'config.json').write_text(json.dumps({
    'format': 'personal-lm-bundle-v1', 'version': f'v{payload["step"]}-cloud',
    'model': payload['config'], 'trained_steps': payload['step'],
    'tokenizer': 'tokenizer.json', 'notes': '个人私有模型；勿分发'}, ensure_ascii=False, indent=1))
shutil.copy(dd / 'tokenizer.json', out / 'tokenizer.json')
(out / 'manifest.json').write_text(json.dumps({'version': f'v{payload["step"]}-cloud', 'private': True}))
print('bundle →', out)
!ls -la {out}

## 回到 mac 的收尾
```sh
# ostore gdrive 同步回 cc 后（或手动从 Drive 下载 exports/<ver>/）
cp exports/<ver>/* "$HOME/Library/Application Support/Keepsake/models/personal-lm/"
# cv-lm 会热加载；/complete 的新 bundle 立即生效
# 注意：98M 模型 d_model=768 与本地 config.json 一致即可被 cvlm-core 加载（layers/d/heads 从 config 读）
```